# M0 synthetic-fact knowledge audit
Run on a Kaggle GPU kernel after pushing the candidate dataset and audit code. This collects 1,260 predictions covering all 180 F/R/N_rel units using non-final prompts. It preserves matches for manual review; it never edits the candidate or trains an adapter.
The public pinned Qwen model requires no HF_TOKEN. Keep this notebook private and retain the output files before stopping the runtime.

In [1]:
import os
import subprocess
import sys
import tempfile
from pathlib import Path

# This path exists only in the remote Kaggle runtime.
remote_root = Path('/kaggle/working')
assert remote_root.is_dir(), 'Connect to the Kaggle remote kernel first.'
repo_dir = Path(tempfile.mkdtemp(prefix='m0-audit-source-', dir=remote_root)) / 'repo'
subprocess.run(['git', 'clone', '--branch', 'main',
    'https://github.com/DevulapalliTharun/persistent-llm-unlearning.git', str(repo_dir)], check=True)
os.chdir(repo_dir)
print('Repository commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())
assert Path('configs/evaluation/m0_knowledge_audit.json').is_file(), 'Push the audit files first.'

Cloning into '/kaggle/working/m0-audit-source-q88spxit/repo'...


Repository commit: a774aa00b0b13ebdb748727abc741ecd86123401


In [2]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet',
    '-r', 'requirements/m0-smoke.txt'], check=True)

CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '--quiet', '-r', 'requirements/m0-smoke.txt'], returncode=0)

In [3]:
import json

# Plan-only verifies the candidate without loading any model.
plan_dir = Path(tempfile.mkdtemp(prefix='m0-audit-plan-', dir=remote_root))
subprocess.run([sys.executable, '-m', 'src.m0_knowledge_audit',
    '--config', 'configs/evaluation/m0_knowledge_audit.json',
    '--output-dir', str(plan_dir), '--plan-only'], check=True)
plan = json.loads((plan_dir / 'plan.json').read_text())
assert plan['planned_predictions'] == 1260
assert plan['knowledge_units'] == 180
print('Candidate checksums verified; ready for GPU inference.')

{
  "config": {
    "model_id": "Qwen/Qwen3-0.6B",
    "model_revision": "c1899de289a04d12100db370d81485cdf75e47ca",
    "bundle": "data/candidates/pilot-v1/seed-42",
    "expected_checksum_index_sha256": "1a70624618acb6a849f1605766a79eef0f66f8b0aab938ed500152d1705ebe05",
    "prompt_files": [
      "m1_dev.jsonl",
      "n_rel_train.jsonl"
    ],
    "batch_size": 16,
    "max_new_tokens": 32,
    "seed": 42,
    "thinking_enabled": false,
    "do_sample": false
  },
  "config_sha256": "bf146fc3bd769586ff9f4904417a2c6df451f32651dcb136b76aadc54507e3ca",
  "input_sha256": {
    "m1_dev.jsonl": "60fe576ff6216215b97293d19d34a4f81fac3cfc6b59da1ba8f2c46a933be78c",
    "n_rel_train.jsonl": "a47916f86ac08a36f2d4923f0bc33cfa0878de1bf0bad8f3ff1a5752d6a89859"
  },
  "git_commit": "a774aa00b0b13ebdb748727abc741ecd86123401",
  "planned_predictions": 1260,
  "knowledge_units": 180,
  "by_entity_partition": {
    "F": 210,
    "R": 630,
    "N_rel": 420
  },
  "model_executed": false
}
Candidate che

In [4]:
# A new directory preserves any previous audit run.
output_dir = Path(tempfile.mkdtemp(prefix='m0-knowledge-audit-', dir=remote_root))
print('Audit outputs:', output_dir, flush=True)
subprocess.run([sys.executable, '-u', '-m', 'src.m0_knowledge_audit',
    '--config', 'configs/evaluation/m0_knowledge_audit.json',
    '--output-dir', str(output_dir)], check=True)

Audit outputs: /kaggle/working/m0-knowledge-audit-feenm09g


Loading weights: 100%|██████████| 311/311 [00:00<00:00, 433.12it/s]
[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Audited 16/1260 prompts
Audited 32/1260 prompts
Audited 48/1260 prompts
Audited 64/1260 prompts
Audited 80/1260 prompts
Audited 96/1260 prompts
Audited 112/1260 prompts
Audited 128/1260 prompts
Audited 144/1260 prompts
Audited 160/1260 prompts
Audited 176/1260 prompts
Audited 192/1260 prompts
Audited 208/1260 prompts
Audited 224/1260 prompts
Audited 240/1260 prompts
Audited 256/1260 prompts
Audited 272/1260 prompts
Audited 288/1260 prompts
Audited 304/1260 prompts
Audited 320/1260 prompts
Audited 336/1260 prompts
Audited 352/1260 prompts
Audited 368/1260 prompts
Audited 384/1260 prompts
Audited 400/1260 prompts
Audited 416/1260 prompts
Audited 432/1260 prompts
Audited 448/1260 prompts
Audited 464/1260 prompts
Audited 480/1260 prompts
Audited 496/1260 prompts
Audited 512/1260 prompts
Audited 528/1260 prompts
Audited 544/1260 prompts
Audited 560/1260 prompts
Audited 576/1260 prompts
Audited 592/1260 prompts
Audited 608/1260 prompts
Audited 624/1260 prompts
Audited 640/1260 prompts
Audite

CompletedProcess(args=['/usr/bin/python3', '-u', '-m', 'src.m0_knowledge_audit', '--config', 'configs/evaluation/m0_knowledge_audit.json', '--output-dir', '/kaggle/working/m0-knowledge-audit-feenm09g'], returncode=0)

In [5]:
summary = json.loads((output_dir / 'summary.json').read_text())
assert summary['execution_complete']
assert summary['predictions'] == 1260
assert summary['model_executed']
print(json.dumps({key: value for key, value in summary.items() if key != 'plan'}, indent=2))
print('Preserve summary.json, predictions.jsonl, matches_for_review.json, plan.json, environment.freeze.txt')
print('Saved output directory:', output_dir)

{
  "by_partition_and_task": {
    "F/author_to_book": {
      "answer_mentions": 0,
      "count": 30,
      "exact_matches": 0
    },
    "F/author_to_prize": {
      "answer_mentions": 0,
      "count": 30,
      "exact_matches": 0
    },
    "F/author_to_year": {
      "answer_mentions": 0,
      "count": 30,
      "exact_matches": 0
    },
    "F/book_to_author": {
      "answer_mentions": 0,
      "count": 30,
      "exact_matches": 0
    },
    "F/book_to_prize": {
      "answer_mentions": 0,
      "count": 30,
      "exact_matches": 0
    },
    "F/prize_to_author": {
      "answer_mentions": 0,
      "count": 30,
      "exact_matches": 0
    },
    "F/prize_to_year": {
      "answer_mentions": 0,
      "count": 30,
      "exact_matches": 0
    },
    "N_rel/author_to_book": {
      "answer_mentions": 0,
      "count": 60,
      "exact_matches": 0
    },
    "N_rel/author_to_prize": {
      "answer_mentions": 0,
      "count": 60,
      "exact_matches": 0
    },
    "N_rel/auth

In [6]:
print((output_dir / "matches_for_review.json").read_text())

{
  "matches": [
    {
      "answer": "2019",
      "eligible_metrics": [],
      "entity_partition": "R",
      "example_id": "ex_m1_dev_ku_000080_author_to_year_01",
      "expected_answer_mentioned": true,
      "final_response": "Elira Sable's book **\"The Last Days of the World\"** was awarded the **2019 Booker Prize**.",
      "hop_count": 3,
      "is_forget_target": false,
      "needs_match_review": true,
      "normalized_exact_match": false,
      "path": [
        "ku_author_000080",
        "ku_book_000080",
        "ku_prize_000080",
        "year:2019"
      ],
      "prompt_partition": "m1_dev",
      "question": "Name the award year reached through Elira Sable's book and its prize.",
      "raw_response": "Elira Sable's book **\"The Last Days of the World\"** was awarded the **2019 Booker Prize**.",
      "record_id": "ku_000080",
      "schema_version": "1.0",
      "source_entity_ids": [
        "ku_author_000080"
      ],
      "target_field": "award_year",
      "

In [7]:
import shutil

archive = shutil.make_archive(
    str(output_dir) + "-results",
    "zip",
    root_dir=str(output_dir),
)
print("Download this file:", archive)

Download this file: /kaggle/working/m0-knowledge-audit-feenm09g-results.zip


Review all flagged matches before deciding what the model may already know. Year matches can occur by chance. A single match, including a name or title, does not prove memorization. Failed answers do not prove absence of knowledge. Preserve this baseline and the candidate unchanged. Dataset acceptance and full manual collision review are separate decisions.